# Part 3 — Evaluation and Interpretation

Notebook này đánh giá một lần các pipeline đã chọn ở Part 2 trên `test.csv`, sau đó phân tích lỗi và các trường hợp hai mô hình bất đồng.

Tập Test chỉ được dùng ở phần đánh giá cuối. Không dùng kết quả Test để điều chỉnh tham số hoặc chọn lại mô hình.

## 1. Chỉ số đánh giá

- **Accuracy:** tỷ lệ email được phân loại đúng trên toàn bộ tập Test.
- **Precision (Spam):** trong các email dự đoán là spam, tỷ lệ thực sự là spam.
- **Recall (Spam):** trong các email spam thật, tỷ lệ được phát hiện.
- **F1 (Spam):** trung bình điều hòa giữa Precision và Recall của lớp spam.
- **Confusion matrix:** hàng là nhãn thật `[Không spam, Spam]`, cột là nhãn dự đoán theo cùng thứ tự.

In [1]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from src.evaluation import (
    calculate_metrics,
    save_confusion_matrix_plots,
    save_metric_comparison,
)

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed").is_dir()
     and (candidate / "artifacts").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
LABEL_NAMES = {0: "Không spam", 1: "Spam"}


## 2. Nạp dữ liệu Test và mô hình đã chọn

Pipeline gồm TF-IDF và classifier được lưu từ Part 2. Cả hai mô hình sẽ dự đoán trên cùng những email Test để phép so sánh công bằng.

In [2]:
test_path = PROCESSED_DIR / "test.csv"
lr_path = ARTIFACTS_DIR / "lr_tuned_pipeline.joblib"
svm_path = ARTIFACTS_DIR / "svm_tuned_pipeline.joblib"
for required_path in (test_path, lr_path, svm_path):
    if not required_path.is_file():
        raise FileNotFoundError(f"Thiếu file cần cho Part 3: {required_path}")

test_df = pd.read_csv(test_path)
required_columns = {"text", "label_num"}
missing = required_columns - set(test_df.columns)
if missing:
    raise ValueError(f"test.csv thiếu cột: {sorted(missing)}")
if test_df["text"].isna().any() or test_df["label_num"].isna().any():
    raise ValueError("test.csv còn giá trị thiếu ở text hoặc label_num.")

X_test = test_df["text"].astype(str)
y_test = test_df["label_num"].astype(int)
models = {
    "Logistic Regression": joblib.load(lr_path),
    "Linear SVM": joblib.load(svm_path),
}
print(f"Số email Test: {len(test_df):,}")
print("Đã nạp hai pipeline tốt nhất từ Part 2.")

Số email Test: 979
Đã nạp hai pipeline tốt nhất từ Part 2.


## 3. Dự đoán một lần và tính chỉ số

Mỗi pipeline được gọi `predict()` một lần trên tập Test. Precision, Recall và F1 được tính cho lớp spam (`pos_label=1`), vì bỏ sót spam là một lỗi quan trọng cần theo dõi.

In [3]:
predictions = {
    model_name: model.predict(X_test)
    for model_name, model in models.items()
}

metrics_df, confusion_matrices = calculate_metrics(y_test, predictions)
display(metrics_df.round(4))
for model_name, matrix in confusion_matrices.items():
    print(f"\nConfusion matrix — {model_name} (hàng thật, cột dự đoán):")
    display(pd.DataFrame(matrix, index=["Thật: Không spam", "Thật: Spam"], columns=["Dự đoán: Không spam", "Dự đoán: Spam"]))

              Model  Accuracy  Precision (Spam)  Recall (Spam)  F1 (Spam)
Logistic Regression    0.9918            0.9897         0.9829     0.9863
         Linear SVM    0.9959            0.9932         0.9932     0.9932

Confusion matrix — Logistic Regression (hàng thật, cột dự đoán):
 Dự đoán: Không spam  Dự đoán: Spam
                 684              3
                   5            287

Confusion matrix — Linear SVM (hàng thật, cột dự đoán):
 Dự đoán: Không spam  Dự đoán: Spam
                 685              2
                   2            290


## 4. Confusion matrix và so sánh trực quan

Mỗi confusion matrix cho biết số email đúng và sai theo từng lớp. Biểu đồ so sánh các chỉ số giúp nhìn nhanh điểm mạnh/yếu trên cùng một Test set.

In [4]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

confusion_figures = save_confusion_matrix_plots(
    confusion_matrices,
    FIGURES_DIR,
    LABEL_NAMES,
)
for model_name, (fig, output_path) in confusion_figures.items():
    display(fig)
    print(f"Đã lưu biểu đồ {model_name}: {output_path}")
    plt.close(fig)

comparison_fig, comparison_plot = save_metric_comparison(metrics_df, FIGURES_DIR)
display(comparison_fig)
print(f"Đã lưu biểu đồ so sánh: {comparison_plot}")
plt.close(comparison_fig)

Figure
Đã lưu biểu đồ Logistic Regression: E:\ML_Classification\results\figures\confusion_matrix_lr.png
Figure
Đã lưu biểu đồ Linear SVM: E:\ML_Classification\results\figures\confusion_matrix_svm.png
Figure
Đã lưu biểu đồ so sánh: E:\ML_Classification\results\figures\model_metrics_comparison.png


## 5. Bất đồng giữa hai mô hình và phân tích lỗi

Ta chia các email thành bốn nhóm: cả hai đúng, chỉ Logistic Regression sai, chỉ Linear SVM sai, và cả hai cùng sai. Các ví dụ văn bản hiển thị được rút gọn để dễ đọc.

In [5]:
comparison_df = test_df[["text", "label_num"]].copy().reset_index(drop=True)
comparison_df["lr_prediction"] = predictions["Logistic Regression"]
comparison_df["svm_prediction"] = predictions["Linear SVM"]
comparison_df["lr_correct"] = comparison_df["lr_prediction"].eq(comparison_df["label_num"])
comparison_df["svm_correct"] = comparison_df["svm_prediction"].eq(comparison_df["label_num"])

both_correct = comparison_df[comparison_df["lr_correct"] & comparison_df["svm_correct"]]
lr_only_wrong = comparison_df[~comparison_df["lr_correct"] & comparison_df["svm_correct"]]
svm_only_wrong = comparison_df[comparison_df["lr_correct"] & ~comparison_df["svm_correct"]]
both_wrong = comparison_df[~comparison_df["lr_correct"] & ~comparison_df["svm_correct"]]
disagreements = comparison_df[comparison_df["lr_prediction"] != comparison_df["svm_prediction"]]

print(f"Cả hai đúng: {len(both_correct)}")
print(f"Chỉ LR sai: {len(lr_only_wrong)}")
print(f"Chỉ SVM sai: {len(svm_only_wrong)}")
print(f"Cả hai cùng sai: {len(both_wrong)}")
print(f"Hai mô hình dự đoán khác nhau: {len(disagreements)}")

def show_examples(frame, title, limit=3):
    print(f"\n{title} — hiển thị tối đa {limit} email")
    if frame.empty:
        print("Không có ví dụ trong nhóm này.")
        return
    sample = frame.head(limit).copy()
    sample["actual"] = sample["label_num"].map(LABEL_NAMES)
    sample["LR"] = sample["lr_prediction"].map(LABEL_NAMES)
    sample["SVM"] = sample["svm_prediction"].map(LABEL_NAMES)
    sample["text_excerpt"] = sample["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 240)
    display(sample[["actual", "LR", "SVM", "text_excerpt"]])

show_examples(disagreements, "Các trường hợp hai mô hình bất đồng", limit=5)
show_examples(lr_only_wrong, "LR sai, SVM đúng")
show_examples(svm_only_wrong, "SVM sai, LR đúng")
show_examples(both_wrong, "Cả hai cùng sai")

Cả hai đúng: 971
Chỉ LR sai: 4
Chỉ SVM sai: 0
Cả hai cùng sai: 4
Hai mô hình dự đoán khác nhau: 4

Các trường hợp hai mô hình bất đồng — hiển thị tối đa 5 email
    actual         LR        SVM                                                                                                                                                                                                                                     text_excerpt
      Spam Không spam       Spam subject: so , it ' s me , varenukha sorry for taking so long . i finally found that site you were asking me about . remember , the one that i used to get a great r \ ate on my homel o an ? i was just looking around the other day and they of
Không spam       Spam Không spam                                                                                                                                                                                            subject: wassup - south park - wassupsouthparkl . exe
      Spam Không 

## 6. Lưu chỉ số và diễn giải điểm SVM

Bảng chỉ số được lưu trong `results/metrics.csv`. Điểm trả về bởi `LinearSVC.decision_function()` biểu thị phía/mức độ cách xa ranh giới phân loại; **đó không phải xác suất**. Logistic Regression có thể trả `predict_proba()`, nhưng xác suất cũng cần được hiệu chỉnh nếu cần diễn giải như độ tin cậy chính xác.

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = RESULTS_DIR / "metrics.csv"
metrics_df.to_csv(metrics_path, index=False, encoding="utf-8")
print(f"Đã lưu bảng chỉ số: {metrics_path}")


Đã lưu bảng chỉ số: E:\ML_Classification\results\metrics.csv
